# Set up the environment

In [36]:
import json
import utils
import pandas as pd
from dotenv import load_dotenv

_ = load_dotenv()

# Set up AISuite

In [37]:
import aisuite as ai

client = ai.Client()

# Set up the database

In [38]:
utils.create_transactions_db()
utils.print_html(utils.get_schema('products.db'))

SQLite database 'products.db' created with a single 'transactions' table (event-sourced).


# Building a SQL generator

## Using an LLM to query a database

In [39]:
def generate_sql(question: str, schema: str, model: str) -> str:
  prompt = f"""
  You are a SQL assistant. Given the schema and the user's question, write a SQL query for SQLite.

  Schema:
  {schema}

  User question:
  {question}

  Respond with the SQL only.
  """

  response = client.chat.completions.create(
    model = model,
    messages = [{ "role": "user", "content": prompt}],
    temperature = 0,
  )

  return response.choices[0].message.content.strip()

In [40]:
schema = """
Table name: transactions
id (INTEGER)
product_id (INTEGER)
product_name (TEXT)
brand (TEXT)
category (TEXT)
color (TEXT)
action (TEXT)
qty_delta (INTEGER)
unit_price (REAL)
notes (TEXT)
ts (DATETIME)
"""

question = "Which color of product has the highest total sales?"

utils.print_html(question, title="User Question")

sql_V1 = generate_sql(question, schema, model="openai:gpt-4.1")

utils.print_html(sql_V1, title="SQL Query V1")

### Validating the output

In [41]:
df_sql_V1 = utils.execute_sql(sql_V1, db_path='products.db')
utils.print_html(df_sql_V1, title="Output of SQL Query V1 - ❌ Does NOT fully answer the question")


color,total_sales
blue,-190571.46


## Refining the SQL query with Reflection

In [42]:
def refine_sql(
  question: str,
  sql_query: str,
  schema: str,
  model: str,
) -> tuple[str, str]:
  """
  Reflect on whether a query's *shown output* answers the question,
  and propose an improved SQL if needed.
  Returns (feedback, refined_sql).
  """

  prompt = f"""
  You are SQL reviewer and refiner.

  User asked:
  {question}

  Original SQL:
  {sql_query}

  Table Schema:
  {schema}

  Step 1: Briefly evaluate if the SQL OUTPUT fully answers the user's question.
  Step 2: If improvement is needed, provide a refined SQL query for SQLite.
  If the original SQL is already correct, return it unchanged.

  Return STRICT JSON with two fields:
  {{
    "feedback": "<1-3 sentences explaining the gap or confirming correctness>",
    "refined_sql": "<final SQL to run>"
  }}
  """

  response = client.chat.completions.create(
    model = model,
    messages = [{ "role": "user", "content": prompt }],
    temperature = 0,
  )

  content = response.choices[0].message.content

  try:
    obj = json.loads(content)
    feedback = str(obj.get("feedback", "")).strip()
    refined_sql = str(obj.get("refined_sql", sql_query)).strip()
    if not refined_sql:
      refineed_sql = sql_query
  except Exception:
    feedback = content.strip()
    refined_sql = sql_query

  return feedback, refined_sql

### Executing the refinement

In [43]:
feedback, sql_V2 = refine_sql(
  question = question,
  sql_query = sql_V1,
  schema = schema,
  model = "openai:gpt-4.1",
)

utils.print_html(question, title = "User Question")
utils.print_html(sql_V1, title = "Generated SQL Query (V1)")

df_sql_V1 = utils.execute_sql(sql_V1, db_path = 'products.db')
utils.print_html(df_sql_V1, title = "SQL Output of V1 - X Does NOT fully answer the question")

utils.print_html(feedback, title = "Feedback on V1")
utils.print_html(sql_V2, title = "Refined SQL Query (V2)")

df_sql_V2 = utils.execute_sql(sql_V2, db_path = 'products.db')
utils.print_html(df_sql_V2, title = "SQL Output of V2 - X Does NOT fully answer the question")

color,total_sales
blue,-190571.46


color,total_sales
blue,-190571.46


## Refining with external feedback

In [44]:
def refine_sql_external_feedback(
  question: str,
  sql_query: str,
  df_feedback: pd.DataFrame,
  schema: str,
  model: str,
) -> tuple[str, str]:
  """
  Evaluate whether the SQL result answers the user's question and,
  if necessary, propose a refined version of the query.
  Returns (feedback, refined_sql).
  """

  prompt = f"""
  You are SQL reviewer and refiner.

  User asked:
  {question}

  Original SQL:
  {sql_query}

  SQL Output:
  {df_feedback.to_markdown(index=False)}

  Table Schema:
  {schema}

  Step 1: Briefly evaluate if the SQL OUTPUT fully answers the user's question.
  Step 2: If improvement is needed, provide a refined SQL query for SQLite.
  If the original SQL is already correct, return it unchanged.

  Return STRICT JSON with two fields:
  {{
    "feedback": "<1-3 sentences explaining the gap or confirming correctness>",
    "refined_sql": "<final SQL to run>"
  }}
  """

  response = client.chat.completions.create(
    model = model,
    messages = [{ "role": "user", "content": prompt }],
    temperature = 0,
  )

  content = response.choices[0].message.content

  try:
    obj = json.loads(content)
    feedback = str(obj.get("feedback", "")).strip()
    refined_sql = str(obj.get("refined_sql", sql_query)).strip()
    if not refined_sql:
      refineed_sql = sql_query
  except Exception:
    feedback = content.strip()
    refined_sql = sql_query

  return feedback, refined_sql

### Executing the refinement with external feedback

In [45]:
df_sql_V1 = utils.execute_sql(sql_V1, db_path='products.db')

feedback, sql_V2 = refine_sql_external_feedback(
    question=question,
    sql_query=sql_V1,
    df_feedback=df_sql_V1,
    schema=schema,
    model="openai:gpt-4.1"
)

utils.print_html(question, title="User Question")
utils.print_html(sql_V1, title="Generated SQL Query (V1)")
utils.print_html(df_sql_V1, title="SQL Output of V1 - ❌ Does NOT fully answer the question")

utils.print_html(feedback, title="Feedback on V1")
utils.print_html(sql_V2, title="Refined SQL Query (V2)")

df_sql_V2 = utils.execute_sql(sql_V2, db_path='products.db')
utils.print_html(df_sql_V2, title="SQL Output of V2 (with External Feedback) - ✅ Fully answers the question")

color,total_sales
blue,-190571.46


color,total_sales
white,358315.09


# The Workflow

In [46]:
def run_sql_workflow(
    db_path: str,
    question: str,
    model_generation: str = "openai:gpt-4.1",
    model_evaluation: str = "openai:gpt-4.1",
):
  """
  End-to-end workflow to generate, execute, evaluate, and refine SQL queries.

  Steps:
    1) Extract database schema
    2) Generate SQL (V1)
    3) Execute V1 → show output
    4) Reflect on V1 with execution feedback → propose refined SQL (V2)
    5) Execute V2 → show final answer
  """

  # 1) Schema
  schema = utils.get_schema(db_path)
  utils.print_html(
      schema,
      title="📘 Step 1 — Extract Database Schema"
  )

  # 2) Generate SQL (V1)
  sql_v1 = generate_sql(question, schema, model_generation)
  utils.print_html(
      sql_v1,
      title="🧠 Step 2 — Generate SQL (V1)"
  )

  # 3) Execute V1
  df_v1 = utils.execute_sql(sql_v1, db_path)
  utils.print_html(
      df_v1,
      title="🧪 Step 3 — Execute V1 (SQL Output)"
  )

  # 4) Reflect on V1 with execution feedback → refine to V2
  feedback, sql_v2 = refine_sql_external_feedback(
      question=question,
      sql_query=sql_v1,
      df_feedback=df_v1,          # external feedback: real output of V1
      schema=schema,
      model=model_evaluation,
  )
  utils.print_html(
      feedback,
      title="🧭 Step 4 — Reflect on V1 (Feedback)"
  )
  utils.print_html(
      sql_v2,
      title="🔁 Step 4 — Refined SQL (V2)"
  )

  # 5) Execute V2
  df_v2 = utils.execute_sql(sql_v2, db_path)
  utils.print_html(
      df_v2,
      title="✅ Step 5 — Execute V2 (Final Answer)"
  )

### Executing the workflow

In [47]:
run_sql_workflow(
    "products.db",
    "Which color of product has the highest total sales?",
    model_generation="openai:gpt-4.1",
    model_evaluation="openai:gpt-4.1"
)

color,total_sales
blue,-190571.46


color,total_sales
white,358315.09
